# Shift-Scan — Google Colab ONLY — 100M-token HARPQ training (resumable)

This notebook is intended to run **only in Google Colab**. It launches **one 100-million-token HARPQ student run** using the existing `shift_scan_advanced_project.zip` source code. It relies on Colab's GPU runtime and Google Drive mount.

**Important reality check:** 100M tokens is a substantial run. The T4 profile below targets a reasonable memory/throughput balance, but it cannot make a T4 train 100M tokens in minutes. The earlier 2M-token run's measured throughput (~920 tokens/s) suggests roughly 30 hours if throughput stays similar; the 10M/2048-token pilot used a different protocol and cannot be used as an exact runtime prediction.

The notebook is designed to:
- use the real pretrained `state-spaces/mamba2-130m` teacher and streamed FineWeb-Edu data;
- run HARPQ (horizon 128) with ternary projections and state distillation;
- save checkpoints/logs to Google Drive;
- resume from `last.pt` after a Colab disconnect;
- record the run configuration and GPU details.

This is a **single HARPQ run**, not a controlled comparison. Do not use it alone to claim HARPQ beats nearest rounding. For that claim, run matched nearest / HARPQ / FP-decay conditions with the same budget and seeds.

## 1. Google Colab setup — mount Drive and locate the source ZIP

**Colab only:** Open this notebook in Google Colab and select `Runtime → Change runtime type → GPU` (T4 or better). Put `shift_scan_advanced_project.zip` in the top level of MyDrive, or change `SOURCE_ZIP` below.

In [ ]:
# Google Colab runtime guard
try:
    import google.colab
except ImportError as exc:
    raise RuntimeError(
        "This notebook is Google Colab ONLY. Open it in Google Colab "
        "and use a GPU runtime."
    ) from exc

from google.colab import drive
drive.mount('/content/drive')

import os, glob, zipfile, shutil, pathlib, json, time, subprocess, sys, textwrap
SOURCE_ZIP = '/content/drive/MyDrive/shift_scan_advanced_project.zip'
assert os.path.exists(SOURCE_ZIP), (
    f'Cannot find {SOURCE_ZIP}. Upload your existing shift_scan_advanced_project.zip '
    'to MyDrive or edit SOURCE_ZIP to its exact Drive path.'
)

WORKDIR = '/content/shift_scan_100m'
os.makedirs(WORKDIR, exist_ok=True)
with zipfile.ZipFile(SOURCE_ZIP) as zf:
    zf.extractall(WORKDIR)

repo_candidates = glob.glob(os.path.join(WORKDIR, '**', 'train_qat.py'), recursive=True)
assert repo_candidates, 'train_qat.py was not found inside the source ZIP.'
REPO = os.path.dirname(repo_candidates[0])
os.chdir(REPO)
print('Using source:', REPO)
print('Files:', sorted(os.listdir(REPO))[:25])

## 2. Install dependencies
This uses the repository's own requirements. If Colab asks for a runtime restart after installation, restart and rerun from Section 1.

In [ ]:
%pip -q install -r requirements.txt
%pip -q install -U 'transformers>=4.40' 'datasets>=2.18' 'huggingface_hub>=0.23'

## 3. Select the 100M run configuration

**T4-safe starting profile:** sequence length 2048, micro-batch 2, gradient accumulation 8 (32,768 tokens per optimizer step). This is a throughput-oriented starting point; if it runs out of memory, change `MICRO_BS=1` and `ACCUM=16` (same effective tokens/step, lower activation memory).

If using an A100/H100, you can increase `MICRO_BS` after a short memory test. Keep the *effective tokens per step* and all settings identical when comparing methods.

In [ ]:
# ---- Main experiment settings ----
RUN_NAME = 'harpq_h128_ternary_100m_seed0'
SEED = 0
TOKEN_BUDGET = 100_007_936  # 3,052 x 32,768 = 100,007,936 tokens

# T4 profile (lower memory fallback: MICRO_BS=1, ACCUM=16)
SEQ_LEN = 2048
MICRO_BS = 2
ACCUM = 8

# HARPQ + ternary student
DECAY = 'shift'
DECAY_QUANTIZER = 'horizon'
DECAY_HORIZON = 128
DECAY_LAG_WEIGHT = 'uniform'
TERNARY = 1
GROUP_SIZE = 128
ACTIVATION_AWARE = 1
STATE_WEIGHT = 0.05
STATE_LOSS_TYPE = 'hybrid'

# T4 uses fp16 here; bf16 is more suitable on newer GPUs that support it well.
DTYPE = 'fp16'
LEARNING_RATE = 1e-4
WARMUP_STEPS = 100
MIN_LR_FRAC = 0.1

# Persistent output in Drive. Checkpoints survive Colab disconnections.
OUT_ROOT = '/content/drive/MyDrive/shift_scan_100m_runs'
OUT_DIR = os.path.join(OUT_ROOT, RUN_NAME)
os.makedirs(OUT_DIR, exist_ok=True)

TOKENS_PER_STEP = SEQ_LEN * MICRO_BS * ACCUM
TOTAL_STEPS = TOKEN_BUDGET // TOKENS_PER_STEP
ACTUAL_TOKENS = TOTAL_STEPS * TOKENS_PER_STEP

print({
    'gpu_profile': 'T4-oriented',
    'gpu': __import__('torch').cuda.get_device_name(0) if __import__('torch').cuda.is_available() else 'NO CUDA',
    'tokens_per_step': TOKENS_PER_STEP,
    'optimizer_steps': TOTAL_STEPS,
    'actual_token_budget': ACTUAL_TOKENS,
    'approx_budget_M': round(ACTUAL_TOKENS / 1e6, 3),
    'output_dir': OUT_DIR
})

## 4. GPU and source checks
Do not start the long run unless CUDA is available and the repository tests pass.

In [ ]:
import torch, subprocess, os, json, platform
assert torch.cuda.is_available(), 'CUDA GPU not found. In Colab choose Runtime > Change runtime type > T4 GPU (or better).'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('Python:', platform.python_version())

# Fast repository correctness check before the expensive training run.
test = subprocess.run([sys.executable, 'tests.py'], text=True)
if test.returncode != 0:
    raise RuntimeError('Repository tests failed; do not launch the 100M run.')

## 5. Estimate runtime from a short throughput pilot

Run a short 200-step pilot first. This is deliberately separate from the 100M run and does not overwrite its checkpoint. The estimate is only a rough guide; if the pilot is unexpectedly slow or unstable, stop before spending the full budget.

In [ ]:
# Optional: run this short pilot before the full 100M experiment.
# Set RUN_PILOT=False only if you have already verified this exact configuration.
RUN_PILOT = True
PILOT_STEPS = 20
if RUN_PILOT:
    pilot_tokens = PILOT_STEPS * TOKENS_PER_STEP
    pilot_dir = os.path.join(OUT_ROOT, RUN_NAME + '_pilot')
    os.makedirs(pilot_dir, exist_ok=True)
    pilot_cmd = [
        sys.executable, 'train_qat.py',
        '--name', RUN_NAME + '_pilot', '--out', OUT_ROOT,
        '--pretrained', 'state-spaces/mamba2-130m',
        '--tokens', str(pilot_tokens), '--seq-len', str(SEQ_LEN),
        '--micro-bs', str(MICRO_BS), '--accum', str(ACCUM),
        '--seed', str(SEED), '--dtype', DTYPE,
        '--lr', str(LEARNING_RATE), '--warmup', str(min(WARMUP_STEPS, PILOT_STEPS)),
        '--decay', DECAY, '--decay-quantizer', DECAY_QUANTIZER,
        '--decay-horizon', str(DECAY_HORIZON), '--decay-lag-weight', DECAY_LAG_WEIGHT,
        '--ternary', str(TERNARY), '--group-size', str(GROUP_SIZE),
        '--activation-aware', str(ACTIVATION_AWARE),
        '--state-weight', str(STATE_WEIGHT), '--state-loss-type', STATE_LOSS_TYPE,
        '--log-every', '20', '--eval-every', str(PILOT_STEPS),
        '--save-every', str(PILOT_STEPS)
    ]
    print('Pilot command:', ' '.join(pilot_cmd))
    t0 = time.time()
    subprocess.run(pilot_cmd, check=True)
    elapsed = time.time() - t0
    measured_tps = pilot_tokens / max(elapsed, 1)
    print(f'Pilot throughput: {measured_tps:,.0f} tokens/s')
    print(f'Rough 100M training-time estimate: {TOKEN_BUDGET / measured_tps / 3600:.1f} hours')
else:
    print('Pilot skipped by user choice.')

## 6. Launch or resume the 100M-token run

The trainer writes `last.pt` every 100 optimizer steps. If Colab disconnects, reconnect, rerun Sections 1–4, then run this cell again. It will automatically resume from the saved checkpoint.

Do not change architecture, quantization, sequence length, batch settings, seed, or dtype when resuming. The trainer validates the model/quantization configuration in the checkpoint.

In [ ]:
CHECKPOINT = os.path.join(OUT_DIR, 'last.pt')
cmd = [
    sys.executable, 'train_qat.py',
    '--name', RUN_NAME, '--out', OUT_ROOT,
    '--pretrained', 'state-spaces/mamba2-130m',
    '--tokens', str(TOKEN_BUDGET), '--seq-len', str(SEQ_LEN),
    '--micro-bs', str(MICRO_BS), '--accum', str(ACCUM),
    '--seed', str(SEED), '--dtype', DTYPE,
    '--lr', str(LEARNING_RATE), '--warmup', str(WARMUP_STEPS),
    '--min-lr-frac', str(MIN_LR_FRAC),
    '--decay', DECAY, '--decay-quantizer', DECAY_QUANTIZER,
    '--decay-horizon', str(DECAY_HORIZON), '--decay-lag-weight', DECAY_LAG_WEIGHT,
    '--ternary', str(TERNARY), '--group-size', str(GROUP_SIZE),
    '--activation-aware', str(ACTIVATION_AWARE),
    '--state-weight', str(STATE_WEIGHT), '--state-loss-type', STATE_LOSS_TYPE,
    '--log-every', '20', '--eval-every', '500', '--save-every', '100',
    '--val-seqs', '32'
]
if os.path.exists(CHECKPOINT):
    cmd += ['--resume', CHECKPOINT]
    print('Existing checkpoint found: RESUMING')
else:
    print('No checkpoint found: starting from step 0')

print('Launching 100M-token HARPQ run...')
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print('Training process completed.')

## 7. Inspect training logs and verify the final checkpoint

In [ ]:
log_path = os.path.join(OUT_DIR, 'log.jsonl')
if os.path.exists(log_path):
    rows = [json.loads(x) for x in open(log_path) if x.strip()]
    print('Log records:', len(rows))
    for row in rows[-10:]:
        print(row)
else:
    print('No log yet:', log_path)

print('Checkpoint exists:', os.path.exists(CHECKPOINT))
if os.path.exists(CHECKPOINT):
    ck = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
    extra = ck.get('extra', {})
    print('Saved step:', extra.get('step'))
    print('Saved tokens:', extra.get('tokens'))

## 8. What this run does — and does not — establish

- It provides a larger HARPQ training run than the 2M-token smoke test.
- It does **not** prove HARPQ is better than nearest rounding.
- For a defensible result, run a matched baseline with `--decay-quantizer nearest`, and a floating-point decay ablation (`--decay exp`), keeping the same data split, token budget, sequence length, effective batch, seed, dtype and evaluation.
- Do not claim speedup from ternary weights: the existing T4 microbenchmarks showed the current Triton ternary GEMV was slower than FP16.
- Keep `args.json`, `run_manifest.json`, `log.jsonl`, `last.pt`, and the final evaluation outputs together on Drive.